In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Only errors are logged
os.environ['TF_GPU_ALLOCATOR'] ='cuda_malloc_async'

import numpy as np
import keras
import matplotlib.pyplot as plt
import tensorflow as tf
from keras import layers
from keras import ops

# TF imports related to tf.data preprocessing
from tensorflow import data as tf_data
from tensorflow import image as tf_image
from tensorflow.keras.utils import plot_model

keras.utils.set_random_seed(42)
from sklearn.model_selection import train_test_split

I0000 00:00:1777036612.043413    2396 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777036616.052952    2396 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [6]:
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 500
SAMPLE_RATE = 16000
OUT_SEQ_LEN = 72000

In [3]:
keras.backend.clear_session(free_memory=True)
train_ds =tf.keras.utils.image_dataset_from_directory(directory='dataset_stft_img_ar/train',
image_size=(300, 300),
batch_size=BATCH_SIZE)
val_ds =tf.keras.utils.image_dataset_from_directory(directory='dataset_stft_img_ar/val',
image_size=(300, 300),
batch_size=BATCH_SIZE)
test_ds =tf.keras.utils.image_dataset_from_directory(directory='dataset_stft_img_ar/test',
image_size=(300, 300),
batch_size=BATCH_SIZE)

Found 880 files belonging to 4 classes.
Found 75 files belonging to 4 classes.
Found 75 files belonging to 4 classes.


In [4]:
model_ar = keras.models.load_model('stft_model.keras')

In [5]:
model_ar.layers

[<Rescaling name=rescaling, built=True>,
 <Functional name=inception_v3, built=True>,
 <GlobalAveragePooling2D name=global_average_pooling2d, built=True>,
 <BatchNormalization name=batch_normalization_94, built=True>,
 <Dense name=dense, built=True>,
 <Dropout name=dropout, built=True>,
 <Dense name=output, built=True>]

In [8]:
model_ar.pop()

<Dense name=output, built=True>

In [10]:
model_ar.add(layers.Dense((NUM_CLASSES),activation = 'softmax',name='output'))

In [14]:
model_ar.trainable=True
model_ar.layers[4].trainable=True
model_ar.layers[6].trainable=True

In [12]:
model_ar.summary()
keras.backend.clear_session(free_memory=True)
model_ar.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-5,weight_decay=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 8, 8, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_94          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,479,286 (101.01 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 22,073,764 (84.20 MB)

 Optimizer params: 4,405,522 (16.81 MB)

In [13]:
model_ar.evaluate(test_ds)

3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - loss: 2.1185 - sparse_categorical_accuracy: 0.2267


[2.118548631668091, 0.2266666740179062]

In [15]:
history_ar = model_ar.fit(train_ds,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
          validation_data=val_ds,
        callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )]        )

Epoch 1/500
28/28 ━━━━━━━━━━━━━━━━━━━━ 67s 980ms/step - loss: 2.1746 - sparse_categorical_accuracy: 0.2682 - val_loss: 1.9525 - val_sparse_categorical_accuracy: 0.2933
Epoch 2/500
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - loss: 1.5462 - sparse_categorical_accuracy: 0.4466 - val_loss: 2.0842 - val_sparse_categorical_accuracy: 0.2933
Epoch 3/500
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - loss: 1.2221 - sparse_categorical_accuracy: 0.5557 - val_loss: 2.1524 - val_sparse_categorical_accuracy: 0.1867
Epoch 4/500
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step - loss: 0.8983 - sparse_categorical_accuracy: 0.6739 - val_loss: 2.0653 - val_sparse_categorical_accuracy: 0.2000
Epoch 5/500
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - loss: 0.8180 - sparse_categorical_accuracy: 0.7159 - val_loss: 1.9217 - val_sparse_categorical_accuracy: 0.2533
Epoch 6/500
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - loss: 0.6725 - sparse_categorical_accuracy: 0.7795 - val_loss: 1.8135 - val_sparse_categorical_accuracy: 0.3067
Ep

In [16]:
model_ar.evaluate(test_ds)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 1.6587 - sparse_categorical_accuracy: 0.3733 


[1.6587437391281128, 0.3733333349227905]